# CICIDS-2017 Dataset Preprocessing

This notebook downloads and preprocesses the **CICIDS-2017** dataset from Kaggle
(`bertvankeulen/cicids-2017`).

The dataset contains ~2.8 million network flow records spread across 8 CSV files
(one per capture day/session). Each file has **78 numerical features** and one
categorical `Label` column covering 14 attack types plus benign traffic.

**Pipeline**
1. Download via `kagglehub`
2. Clean data (inf / NaN, whitespace in column names, duplicates)
3. Encode the label column
4. Drop low-variance / redundant features
5. Normalise numerical features
6. Stream-write to CSV + JSON-Lines
7. Download outputs (Colab)


In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler

print('Libraries imported.')

## 1. Download the dataset

In [ ]:
print('Downloading dataset...')
data_path = kagglehub.dataset_download('bertvankeulen/cicids-2017')
print(f'Dataset ready at: {data_path}')

print('\nFiles in dataset:')
all_files = sorted(
    [f for f in os.listdir(data_path) if f.endswith('.csv')]
)
for f in all_files:
    print(f'  {f}')

## 2. Configuration

In [ ]:
OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# CICIDS-2017 target column (may have leading/trailing spaces in raw files)
TARGET_COL = 'Label'

# Columns to drop: identifiers that leak information or are not useful features
# The dataset contains a duplicate 'Fwd Header Length' column — we drop one copy.
DROP_COLS = [
    'Flow ID',
    ' Source IP',
    ' Source Port',
    ' Destination IP',
    ' Destination Port',
    ' Timestamp',
    'Fwd Header Length.1',   # duplicate column present in some files
]

print('Config set.')

## 3. Preprocessing functions

In [ ]:
def strip_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Remove leading/trailing whitespace from column names."""
    df.columns = df.columns.str.strip()
    return df


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Handle infinities, NaNs, duplicates and normalise the label."""
    df = df.copy()

    # Replace infinite values with NaN then forward-fill / drop
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Drop rows where all feature columns are NaN
    df.dropna(how='all', inplace=True)

    # Fill remaining NaNs in numeric columns with column median
    num_cols = df.select_dtypes(include='number').columns.tolist()
    if TARGET_COL in num_cols:
        num_cols.remove(TARGET_COL)
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    # Remove exact duplicate rows
    df.drop_duplicates(inplace=True)

    # Normalise label: strip whitespace, title-case for consistency
    df[TARGET_COL] = (
        df[TARGET_COL]
        .astype(str)
        .str.strip()
        .str.title()           # e.g. 'dos hulk' → 'Dos Hulk'
        .replace({'Web Attack \x96 Brute Force': 'Web Attack Brute Force',
                  'Web Attack \x96 Xss':          'Web Attack Xss',
                  'Web Attack \x96 Sql Injection': 'Web Attack Sql Injection'})
    )

    return df


def drop_unwanted_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Remove identifier and duplicate columns."""
    cols_to_drop = [c for c in DROP_COLS if c in df.columns]
    return df.drop(columns=cols_to_drop)


def downcast_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Reduce memory usage by downcasting numeric columns."""
    for col in df.select_dtypes(include='float64').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include='int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df


print('Functions defined.')

## 4. Fit encoder & scaler on the first file

In [ ]:
label_encoder: LabelEncoder = LabelEncoder()
scaler: StandardScaler = StandardScaler()
numeric_cols: list = []


def fit_transformers(df: pd.DataFrame) -> None:
    """Fit LabelEncoder and StandardScaler on the given dataframe."""
    global numeric_cols

    # All columns except the label are treated as numerical for CICIDS-2017
    numeric_cols = [c for c in df.columns if c != TARGET_COL]

    # Fit label encoder on known classes
    label_encoder.fit(df[TARGET_COL])
    print(f'  Classes found: {list(label_encoder.classes_)}')

    # Fit scaler on numeric features
    scaler.fit(df[numeric_cols])
    print('  Scaler fitted.')


def encode_and_scale(df: pd.DataFrame) -> pd.DataFrame:
    """Apply fitted encoder and scaler to a dataframe."""
    df = df.copy()

    # Encode label — map unseen labels to 'BENIGN' (index 0 fallback)
    df[TARGET_COL] = df[TARGET_COL].apply(
        lambda x: x if x in label_encoder.classes_ else label_encoder.classes_[0]
    )
    df[TARGET_COL] = label_encoder.transform(df[TARGET_COL])

    # Scale numeric features
    df[numeric_cols] = scaler.transform(df[numeric_cols])

    return df


print('Transformer helpers defined.')

## 5. Process all files and stream output

In [ ]:
csv_out  = os.path.join(OUTPUT_DIR, 'cicids2017_processed.csv')
json_out = os.path.join(OUTPUT_DIR, 'cicids2017_processed.jsonl')

# Clear any previous outputs
for f in [csv_out, json_out]:
    if os.path.exists(f):
        os.remove(f)

first_file = True
total_rows = 0

for idx, filename in enumerate(all_files, start=1):
    path = os.path.join(data_path, filename)
    print(f'\nProcessing file {idx}/{len(all_files)}: {filename}')

    # ── Load ────────────────────────────────────────────────────────────────
    df = pd.read_csv(path, low_memory=False)
    print(f'  Loaded: {len(df):,} rows, {df.shape[1]} columns')

    # ── Clean ───────────────────────────────────────────────────────────────
    df = strip_column_names(df)
    df = drop_unwanted_cols(df)
    df = clean_data(df)
    df = downcast_dtypes(df)
    print(f'  After cleaning: {len(df):,} rows')

    # ── Fit on first file, then encode & scale ──────────────────────────────
    if first_file:
        fit_transformers(df)

    df = encode_and_scale(df)

    # ── Stream to disk ──────────────────────────────────────────────────────
    df.to_csv(csv_out,  mode='a', header=first_file, index=False)
    df.to_json(json_out, orient='records', lines=True, mode='a')

    total_rows += len(df)
    print(f'  ✅ Written — running total: {total_rows:,} rows')

    del df
    first_file = False

print(f'\n{"="*55}')
print(f'✅  Processing complete!')
print(f'   Total rows : {total_rows:,}')
print(f'   CSV        → {csv_out}')
print(f'   JSONL      → {json_out}')

## 6. Quick sanity check

In [ ]:
sample = pd.read_csv(csv_out, nrows=5)
print('Shape of first 5 rows loaded back:')
print(sample.shape)
print()
print(sample.head())
print()
print('Dtypes:')
print(sample.dtypes)
print()
print('Encoded label classes:')
for i, cls in enumerate(label_encoder.classes_):
    print(f'  {i} → {cls}')

## 7. Download outputs (Colab only)

In [ ]:
try:
    from google.colab import files
    print('Downloading CSV...')
    files.download(csv_out)
    print('Downloading JSONL...')
    files.download(json_out)
except ImportError:
    print('Not running in Colab — files saved locally:')
    print(f'  {csv_out}')
    print(f'  {json_out}')